# Unsupervised Pair Ranking


In [10]:
import re
import numpy as np
from pathlib import Path
from collections import defaultdict

TXT_PATH = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")
OUT_PATH = Path("/home/ac.zzheng/power/GPGPU/data/H100/pred_rows.txt")
FEATURE_KEYS = ["dram_min_scaled", "sm_min_scaled", "fp_min_scaled"]


SEED_W_DRAM = np.array([0.8, 0.10, 0.1], dtype=float)
SEED_W_SM = np.array([0.1, 0.8, 0.1], dtype=float)
SEED_W_BAL = np.array([0.33, 0.33, 0.33], dtype=float)

SEED_B = 0.0
LR = 0.03
EPOCHS = 800
L2_SEED = 0.10
TIE_THRESH = 0

# Seed selector config
SEED_SELECTOR_CFG = {
    "corr_margin": 0.20,
    "cv_ratio_hi": 1.20,
    "cv_ratio_lo": 0.80,
    "enable_sm_branch": True,
}


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def parse_txt_9col(path):
    lines = path.read_text().splitlines()
    rows = []
    suite = app = better = None
    cap = None
    in_table = False

    header = (
        "gpu_count\tperformance\tdram_sum\tsm_sum\tfp_sum\t"
        "dram_min_scaled\tsm_min_scaled\tfp_min_scaled"
    )

    for ln in lines:
        if ln.startswith("===== ") and ln.endswith(" ====="):
            m = re.match(r"^=====\s+(\w+)/(.*?)\s+=====$", ln)
            if m:
                suite = m.group(1).strip().upper()
                app = m.group(2).strip()
            continue

        if ln.startswith("(performance better when "):
            better = "lower" if "lower" in ln else "higher"
            continue

        m = re.match(r"^cap=(\d+)\s+true_rank=\[(.*?)\]$", ln)
        if m:
            cap = int(m.group(1))
            in_table = False
            continue

        if ln.startswith(header):
            in_table = True
            continue

        if in_table and ln.strip():
            ps = ln.split("\t")
            if len(ps) == 8 and ps[0].isdigit():
                rows.append({
                    "suite": suite,
                    "app": app,
                    "cap": cap,
                    "gpu": int(ps[0]),
                    "perf": float(ps[1]),
                    "dram_min_scaled": float(ps[5]),
                    "sm_min_scaled": float(ps[6]),
                    "fp_min_scaled": float(ps[7]),
                    "better": better,
                })
            else:
                in_table = False

    by_case = defaultdict(list)
    for r in rows:
        by_case[(r["suite"], r["app"], r["cap"])].append(r)
    return by_case


def minmax_norm(vals):
    lo, hi = min(vals), max(vals)
    if hi - lo < 1e-12:
        return [0.0] * len(vals)
    return [(v - lo) / (hi - lo) for v in vals]


def build_case_feats(rows):
    feat_cols = [minmax_norm([r[k] for r in rows]) for k in FEATURE_KEYS]
    feats = []
    for i, r in enumerate(rows):
        vec = np.array([feat_cols[j][i] for j in range(len(FEATURE_KEYS))], dtype=float)
        feats.append({
            "gpu": r["gpu"],
            "perf": r["perf"],
            "feat": vec,
            "better": r["better"],
            "dram_min_scaled": r["dram_min_scaled"],
            "sm_min_scaled": r["sm_min_scaled"],
            "fp_min_scaled": r["fp_min_scaled"],
        })
    return feats


def rankdata_avg_tie(vals):
    vals = np.asarray(vals, dtype=float)
    order = np.argsort(vals)
    ranks = np.empty(len(vals), dtype=float)
    i = 0
    while i < len(vals):
        j = i
        while j + 1 < len(vals) and vals[order[j + 1]] == vals[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return ranks


def spearman_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or np.allclose(x, x[0]) or np.allclose(y, y[0]):
        return 0.0
    rx = rankdata_avg_tie(x)
    ry = rankdata_avg_tie(y)
    rx -= rx.mean()
    ry -= ry.mean()
    den = np.sqrt((rx * rx).sum() * (ry * ry).sum())
    if den <= 1e-12:
        return 0.0
    return float((rx * ry).sum() / den)


def safe_ratio_max_min(vals):
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return 1.0
    vmin = float(vals.min())
    vmax = float(vals.max())
    if vmin <= 1e-12:
        return float("inf") if vmax > 1e-12 else 1.0
    return vmax / vmin


def safe_cv(vals):
    vals = np.asarray(vals, dtype=float)
    m = float(vals.mean())
    if abs(m) <= 1e-12:
        return 0.0
    return float(vals.std() / abs(m))


def collect_profile_stats(feats):
    g = np.array([x["gpu"] for x in feats], dtype=float)
    dram = np.array([x["dram_min_scaled"] for x in feats], dtype=float)
    sm = np.array([x["sm_min_scaled"] for x in feats], dtype=float)
    fp = np.array([x["fp_min_scaled"] for x in feats], dtype=float)

    rho_dram = spearman_corr(g, dram)
    rho_sm = spearman_corr(g, sm)

    stats = {
        "rho_dram": rho_dram,
        "rho_sm": rho_sm,
        "abs_rho_dram": abs(rho_dram),
        "abs_rho_sm": abs(rho_sm),
        "cv_dram": safe_cv(dram),
        "cv_sm": safe_cv(sm),
        "cv_ratio": safe_cv(dram) / (safe_cv(sm) + 1e-9),
        "fp_flat": safe_ratio_max_min(fp),
        "sm_spread": safe_ratio_max_min(sm),
    }
    return stats


def select_seed_from_profile(feats, cfg=SEED_SELECTOR_CFG):
    """
    CV-ratio-only selector (3 policies):
      - dram_tiebreak
      - sm_tiebreak
      - balanced
    """
    stats = collect_profile_stats(feats)

    cvr = stats["cv_ratio"]

    # DRAM-dominant zone
    if cvr >= 1.10:
        stats["seed_policy"] = "dram_tiebreak"
        return np.array(SEED_W_DRAM, dtype=float), stats

    # SM-dominant zone (moderate low only)
    if 0.55 <= cvr <= 0.95:
        stats["seed_policy"] = "sm_tiebreak"
        return np.array(SEED_W_SM, dtype=float), stats

    # Very low cv_ratio (<0.55) or near-1 uncertain zone (0.95~1.10)
    stats["seed_policy"] = "balanced"
    return np.array(SEED_W_BAL, dtype=float), stats



def build_pseudo_pairs(feats, seed_w, seed_b):
    X, y = [], []
    seed_scores = [float(seed_w @ x["feat"] + seed_b) for x in feats]
    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            diff = feats[i]["feat"] - feats[j]["feat"]
            yi = 1.0 if seed_scores[i] > seed_scores[j] else 0.0
            X += [diff, -diff]
            y += [yi, 1.0 - yi]
    return np.array(X, dtype=float), np.array(y, dtype=float)


def train_case_unsup(X, y, w0, b0=0.0, lr=0.03, epochs=800, l2_seed=0.10):
    w = np.array(w0, dtype=float).copy()
    b = float(b0)
    n = len(y)
    if n == 0:
        return w, b
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        grad_w = (X.T @ (p - y)) / n + l2_seed * (w - w0)
        grad_b = np.mean(p - y)
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b


def rank_case(feats, w, b):
    scores = np.zeros(len(feats), dtype=float)
    for i in range(len(feats)):
        for j in range(len(feats)):
            if i == j:
                continue
            scores[i] += sigmoid((feats[i]["feat"] - feats[j]["feat"]) @ w + b)
    order = np.argsort(-scores)
    return [feats[k]["gpu"] for k in order]


def true_rank(feats, better):
    if better == "lower":
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"])
    else:
        idx = sorted(range(len(feats)), key=lambda i: feats[i]["perf"], reverse=True)
    return [feats[i]["gpu"] for i in idx]


def top1_true_set(feats, better, tie_thresh=0.03):
    ranked = sorted(feats, key=lambda x: x["perf"], reverse=(better != "lower"))
    best = ranked[0]["perf"]
    if len(ranked) == 1 or best <= 0:
        return {ranked[0]["gpu"]}
    second = ranked[1]["perf"]
    gap = (second - best) / best if better == "lower" else (best - second) / best
    return {ranked[0]["gpu"], ranked[1]["gpu"]} if gap <= tie_thresh else {ranked[0]["gpu"]}


def perf_diff_pct(feats, gpu_a, gpu_b):
    perf_dict = {x["gpu"]: x["perf"] for x in feats}
    p = perf_dict.get(gpu_a)
    t = perf_dict.get(gpu_b)
    if p is None or t is None or abs(t) < 1e-12:
        return float("nan")
    return (p - t) / abs(t) * 100.0


def format_pred_row(app, cap, pred, trank, hit, feats):
    """Format a prediction row for file output.
    MISS: perf_diff between predicted top1 vs true top1.
    OK:   perf_diff between true top1 vs true top2 if top1 uses more GPUs.
    """
    marker = "OK" if hit else "MISS"
    base = "{:<12s} cap={:4d} [{:4s}] pred={} true={}".format(
        app, cap, marker, pred, trank)
    if not hit:
        diff_pct = perf_diff_pct(feats, pred[0], trank[0])
        return base + " perf_diff={:+.2f}%".format(diff_pct)
    if len(trank) >= 2 and trank[0] > trank[1]:
        diff_pct = perf_diff_pct(feats, trank[0], trank[1])
        return base + " perf_diff={:+.2f}%".format(diff_pct)
    return base


def run_eval(target_suite, by_case, fixed_seed=None):
    cases = [(k, v) for k, v in sorted(by_case.items()) if k[0] == target_suite]
    print("\n{} cases: {}".format(target_suite, len(cases)))
    print("FEATURE_KEYS:", FEATURE_KEYS)
    if fixed_seed is not None:
        print("FIXED_SEED:", fixed_seed.tolist())
    else:
        print("SEED_W_DRAM:", SEED_W_DRAM.tolist())
        print("SEED_W_SM:", SEED_W_SM.tolist())
        print("SEED_W_BAL:", SEED_W_BAL.tolist())
        print("SEED_SELECTOR_CFG:", SEED_SELECTOR_CFG)

    hits = 0
    results = []
    weights = []
    policy_counts = defaultdict(int)
    policy_hits = defaultdict(int)

    for (suite, app, cap), rows in cases:
        feats = build_case_feats(rows)

        if fixed_seed is not None:
            seed_w = np.array(fixed_seed, dtype=float)
            stats = collect_profile_stats(feats)
            stats["seed_policy"] = "fixed"
        else:
            seed_w, stats = select_seed_from_profile(feats)

        X, y = build_pseudo_pairs(feats, seed_w, SEED_B)
        w_case, b_case = train_case_unsup(X, y, w0=seed_w, b0=SEED_B, lr=LR, epochs=EPOCHS, l2_seed=L2_SEED)

        pred = rank_case(feats, w_case, b_case)
        trank = true_rank(feats, feats[0]["better"])
        tset = top1_true_set(feats, feats[0]["better"], tie_thresh=TIE_THRESH)
        hit = int(pred[0] in tset)

        hits += hit
        policy = stats["seed_policy"]
        policy_counts[policy] += 1
        policy_hits[policy] += hit

        results.append((app, cap, pred, trank, hit, stats, feats))
        weights.append(w_case)

    acc = hits / len(cases) if cases else 0.0
    print("\nTop-1 accuracy ({}): {:.4f} ({}/{})".format(target_suite, acc, hits, len(cases)))

    print("\nPolicy usage / accuracy:")
    for p in ["fixed", "dram_tiebreak", "sm_tiebreak", "balanced"]:
        n = policy_counts[p]
        if n > 0:
            print("  {:14s}: {:3d} cases, acc {:.4f} ({}/{})".format(
                p, n, policy_hits[p] / n, policy_hits[p], n
            ))

    print("\nAverage learned per-case weights:")
    if len(weights):
        W = np.vstack(weights)
        for i, k in enumerate(FEATURE_KEYS):
            print("w_{} = {}".format(k, round(float(W[:, i].mean()), 6)))

    print("\nPred vs True (all cases):")
    for app, cap, pred, trank, hit, stats, feats in results[0:-1]:
        marker = "OK" if hit else "MISS"
        print(
            "{:<12s} cap={:4d} [{:4s}] pred={} true={}".format(
                app, cap,
                marker, pred, trank,
            )
        )

    print("\nMispredictions:")
    for app, cap, pred, trank, hit, stats, feats in results:
        if not hit:
            diff_pct = perf_diff_pct(feats, pred[0], trank[0])
            print(
                "{:<12s} cap={:4d} policy={:14s} "
                "rhoD={:+.3f} rhoS={:+.3f} cvR={:.3f} perf_diff={:+.2f}% pred={} true={}".format(
                    app,
                    cap,
                    stats["seed_policy"],
                    stats["rho_dram"],
                    stats["rho_sm"],
                    stats["cv_ratio"],
                    diff_pct,
                    pred,
                    trank,
                )
            )

    return results


by_case = parse_txt_9col(TXT_PATH)
print("Parsed total cases:", len(by_case))

spec_results = run_eval("SPEC", by_case)

ml_results = run_eval("ML", by_case)
with open(OUT_PATH, "w") as f:
    for suite_name, suite_results in [("SPEC", spec_results), ("ML", ml_results)]:
        for app, cap, pred, trank, hit, stats, feats in suite_results:
            f.write(format_pred_row(app, cap, pred, trank, hit, feats) + "\n")
print("Predictions written to", OUT_PATH)

Parsed total cases: 135

SPEC cases: 90
FEATURE_KEYS: ['dram_min_scaled', 'sm_min_scaled', 'fp_min_scaled']
SEED_W_DRAM: [0.8, 0.1, 0.1]
SEED_W_SM: [0.1, 0.8, 0.1]
SEED_W_BAL: [0.33, 0.33, 0.33]
SEED_SELECTOR_CFG: {'corr_margin': 0.2, 'cv_ratio_hi': 1.2, 'cv_ratio_lo': 0.8, 'enable_sm_branch': True}

Top-1 accuracy (SPEC): 0.9333 (84/90)

Policy usage / accuracy:
  dram_tiebreak :  21 cases, acc 0.9524 (20/21)
  sm_tiebreak   :  32 cases, acc 0.9688 (31/32)
  balanced      :  37 cases, acc 0.8919 (33/37)

Average learned per-case weights:
w_dram_min_scaled = 1.066992
w_sm_min_scaled = 0.98712
w_fp_min_scaled = 0.73754

Pred vs True (all cases):
cloverleaf   cap= 400 [OK  ] pred=[1, 2] true=[1, 2]
cloverleaf   cap= 500 [OK  ] pred=[2, 1] true=[2, 1]
cloverleaf   cap= 600 [OK  ] pred=[2, 1, 3] true=[2, 3, 1]
cloverleaf   cap= 700 [OK  ] pred=[2, 3, 1] true=[2, 3, 1]
cloverleaf   cap= 800 [OK  ] pred=[3, 2, 4, 1] true=[3, 2, 4, 1]
cloverleaf   cap= 900 [OK  ] pred=[3, 4, 2, 1] true=[3, 4,